In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import ruptures as rpt
import multiprocessing
from concurrent import futures
from tqdm import tqdm

import sys

sys.path.append("..")

from src import IOFunctions
from IPython.display import display, clear_output

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter(dark_background=False)

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

In [ ]:
dye_data_folder = '/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/Brendan/20260202_HollidayJunctions/200mMMgCl2'

In [ ]:
loc_files = H_F.file_search(dye_data_folder, ".h5", "")
print(f"Found {len(loc_files)} H5 files")
for i, f in enumerate(loc_files):
    print(f"  [{i}]: {os.path.basename(f)}")

In [ ]:
notch_filter = "semrock-nf03-405-488-561-635e"
dichroic_mirror = "semrock-di03-r488-561-t1-25x36"
dichroic_mirror_all = "semrock-di03-r405-488-561-635-t1-25x36"
shortpass_filter = "semrock-bsp01-785r"
longpass_red_filter = "semrock-blp01-635r"

longpass_filter = "semrock-blp01-568r"
filters = [dichroic_mirror]
dye = "Cy3"
average_emission_wavelengths, dye_pixel_efficiency = (
    S_F.get_pixel_fractions_dye_and_filters(dye, filters, wavelength, pixel_QYs)
)
dye_pixel_efficiency_Cy3 = dye_pixel_efficiency / np.sum(dye_pixel_efficiency)

dye = "Cy5"
average_emission_wavelengths, dye_pixel_efficiency = (
    S_F.get_pixel_fractions_dye_and_filters(dye, filters, wavelength, pixel_QYs)
)
dye_pixel_efficiency_Cy5 = dye_pixel_efficiency / np.sum(dye_pixel_efficiency)

print(f"Cy3 pixel efficiencies (B, G, R): {dye_pixel_efficiency_Cy3}")
print(f"Cy5 pixel efficiencies (B, G, R): {dye_pixel_efficiency_Cy5}")

In [ ]:
def filter_initial_guess(df, tol=0.01):
    """Remove datapoints where A_R, A_G, and A_B are all within tol of 0.33.
    
    These represent the initial guess that was never updated by the fitter.
    
    Args:
        df (pd.DataFrame): DataFrame with A_R, A_G, A_B columns.
        tol (float): Tolerance for comparison to 0.33.
    
    Returns:
        pd.DataFrame: Filtered DataFrame with initial guesses removed.
    """
    initial_guess_mask = (
        np.isclose(df['A_R'], 0.33, atol=tol) &
        np.isclose(df['A_G'], 0.33, atol=tol) &
        np.isclose(df['A_B'], 0.33, atol=tol)
    )
    n_removed = initial_guess_mask.sum()
    n_total = len(df)
    print(f"Removed {n_removed}/{n_total} initial guess datapoints ({100*n_removed/n_total:.1f}%)")
    return df[~initial_guess_mask].copy()

In [ ]:
def find_colour_CPs(A_R, A_G, model="l2", min_size=5, jump=1):
    """Find change points jointly in A_R and A_G using multivariate detection.

    A_R and A_G are anti-correlated spectral fractions (A_R + A_G + A_B = 1),
    so a FRET transition shifts both simultaneously. Joint detection captures
    this correlated change rather than treating channels independently.

    Uses BIC penalty (log(n) * sigma^2) rather than the n * sigma^2 used for
    photoelectron traces — spectral fractions are bounded [0, 1] so the
    original penalty is orders of magnitude too conservative.

    Args:
        A_R (np.ndarray): Red spectral fraction time series.
        A_G (np.ndarray): Green spectral fraction time series.
        model (str): Cost model for ruptures.
        min_size (int): Minimum segment size.
        jump (int): Grid of change point candidates.

    Returns:
        list: Change point indices (last element is always len(signal)).
    """
    n = len(A_R)
    if n < min_size:
        return [n]
    # stack into 2D array (n_samples, 2) for multivariate detection
    signal = np.column_stack([A_R, A_G])
    dim = signal.shape[1]
    # estimate noise from per-channel variance
    sigma2 = np.nanmean([np.nanvar(A_R), np.nanvar(A_G)])
    if sigma2 == 0:
        return [n]
    # BIC penalty: log(n) * dim * sigma^2
    pen = np.log(n) * dim * sigma2
    algo = rpt.Pelt(model=model, min_size=min_size, jump=jump).fit(signal)
    my_CPs = algo.predict(pen=pen)
    return my_CPs


def find_CPs_for_puncta(args):
    """Find change points for a single punctum jointly on A_R and A_G.

    Args:
        args (tuple): (puncta_id, A_R_values, A_G_values)

    Returns:
        tuple: (puncta_id, CPs, has_changepoint)
    """
    puncta_id, A_R, A_G = args
    CPs = find_colour_CPs(A_R, A_G)
    # has changepoint if more than 1 CP
    # (ruptures always reports the final index as a CP)
    has_cp = len(CPs) > 1
    return (puncta_id, CPs, has_cp)


def find_CPs_parallel(df):
    """Run joint change point detection on all puncta in parallel.

    Args:
        df (pd.DataFrame): Filtered DataFrame with puncta_id, A_R, A_G.

    Returns:
        dict: {puncta_id: CPs} for puncta with at least one change point.
    """
    puncta_ids = df['puncta_id'].unique()

    # prepare tasks
    tasks = []
    for pid in puncta_ids:
        subset = df[df['puncta_id'] == pid].sort_values('frame')
        A_R = subset['A_R'].values.astype(np.float64)
        A_G = subset['A_G'].values.astype(np.float64)
        tasks.append((pid, A_R, A_G))

    n_workers = min(60, max(1, int(0.9 * multiprocessing.cpu_count())))
    cp_results = {}

    with futures.ProcessPoolExecutor(n_workers) as executor:
        fs = {executor.submit(find_CPs_for_puncta, task): task[0] for task in tasks}
        with tqdm(desc="Finding colour CPs", total=len(tasks), unit="punctum") as pbar:
            for f in futures.as_completed(fs):
                pbar.update()
                puncta_id, CPs, has_cp = f.result()
                if has_cp:
                    cp_results[puncta_id] = CPs

    print(f"\nPuncta with change points: {len(cp_results)}/{len(puncta_ids)} "
          f"({100*len(cp_results)/len(puncta_ids):.1f}%)")
    return cp_results

## Load data and apply filters

In [ ]:
# load data - change index as needed
file_idx = 1
print(f"Loading: {os.path.basename(loc_files[file_idx])}")
df = pd.read_hdf(loc_files[file_idx])
print(f"Total datapoints: {len(df)}")
print(f"Unique puncta: {df['puncta_id'].nunique()}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# step 1: remove datapoints where A_R, A_G, A_B are all ~0.33 (initial guess)
df_filtered = filter_initial_guess(df, tol=0.01)
print(f"Remaining datapoints: {len(df_filtered)}")
print(f"Remaining puncta: {df_filtered['puncta_id'].nunique()}")

In [ ]:
# step 2: find change points in A_R and A_G channels
cp_results = find_CPs_parallel(df_filtered)

In [ ]:
# step 3: keep only puncta with at least one change point
cp_puncta_ids = list(cp_results.keys())
df_cp = df_filtered[df_filtered['puncta_id'].isin(cp_puncta_ids)].copy()
print(f"Puncta with change points: {len(cp_puncta_ids)}")
print(f"Datapoints in filtered dataset: {len(df_cp)}")

## Visualise puncta with change points

In [ ]:
exposure_time = 0.1  # seconds per frame

for pid in sorted(cp_puncta_ids):
    subset = df_cp[df_cp['puncta_id'] == pid].sort_values('frame')
    if len(subset) < 5:
        continue

    CPs = cp_results[pid]
    t = subset['frame'].values * exposure_time
    ratio = subset['A_R'].values / subset['A_G'].values
    photons = subset['photons'].values

    fig, axs = plotter.two_column_plot(ncolumns=3, widthratio=[1, 1, 1])
    fig.suptitle(f"Puncta {pid}  ({len(CPs)-1} change point(s))", fontsize=10)

    # segment boundaries: [0, cp1, cp2, ..., len(signal)]
    boundaries = [0] + list(CPs)

    # panel 1: A_R and A_G vs time with change points
    axs[0].scatter(t, subset['A_R'].values, s=5, color='red', label='A_R')
    axs[0].scatter(t, subset['A_G'].values, s=5, color='green', label='A_G')
    axs[0].scatter(t, subset['A_B'].values, s=5, color='blue', label='A_B', alpha=0.3)
    for cp_idx in CPs[:-1]:
        if cp_idx < len(t):
            axs[0].axvline(x=t[cp_idx], color='black', ls='--', alpha=0.7, lw=1.5)
    # segment-mean spectral fractions
    for i in range(len(boundaries) - 1):
        seg_start = boundaries[i]
        seg_end = min(boundaries[i + 1], len(t))
        if seg_end <= seg_start:
            continue
        t_start = t[seg_start]
        t_end = t[seg_end - 1]
        axs[0].hlines(y=np.nanmean(subset['A_R'].values[seg_start:seg_end]), xmin=t_start, xmax=t_end, color='darkred', ls='-', lw=2, alpha=0.8)
        axs[0].hlines(y=np.nanmean(subset['A_G'].values[seg_start:seg_end]), xmin=t_start, xmax=t_end, color='darkgreen', ls='-', lw=2, alpha=0.8)
        axs[0].hlines(y=np.nanmean(subset['A_B'].values[seg_start:seg_end]), xmin=t_start, xmax=t_end, color='darkblue', ls='-', lw=2, alpha=0.5)
    axs[0].set_xlabel('time / s')
    axs[0].set_ylabel('spectral fraction')
    axs[0].set_ylim([0, 1])
    axs[0].legend(fontsize=6, markerscale=2)

    # panel 2: R/G ratio vs time with segment averages
    axs[1] = plotter.scatter_plot(
        axs[1], x=t, y=ratio, xaxislabel='time / s',
        yaxislabel='R/G ratio', facecolor='red'
    )
    for cp_idx in CPs[:-1]:
        if cp_idx < len(t):
            axs[1].axvline(x=t[cp_idx], color='black', ls='--', alpha=0.7, lw=1.5)
    # segment-mean R/G ratio
    for i in range(len(boundaries) - 1):
        seg_start = boundaries[i]
        seg_end = min(boundaries[i + 1], len(t))
        if seg_end <= seg_start:
            continue
        seg_mean = np.nanmean(ratio[seg_start:seg_end])
        t_start = t[seg_start]
        t_end = t[seg_end - 1]
        axs[1].hlines(y=seg_mean, xmin=t_start, xmax=t_end, color='green', ls='-', lw=2, alpha=0.8)
    # reference lines for Cy3/Cy5
    xmin, xmax = axs[1].get_xlim()
    axs[1].hlines(
        y=dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1],
        xmin=xmin, xmax=xmax, ls='--', color='darkred', label='Cy5'
    )
    axs[1].hlines(
        y=dye_pixel_efficiency_Cy3[2] / dye_pixel_efficiency_Cy3[1],
        xmin=xmin, xmax=xmax, ls='--', color='orange', label='Cy3'
    )
    axs[1].set_xlim([xmin, xmax])
    ymin, ymax = axs[1].get_ylim()
    if dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1] > ymax:
        axs[1].set_ylim(ymin, 1.1 * (dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1]))
    axs[1].legend(fontsize=6)

    # panel 3: photons vs time with segment averages
    axs[2] = plotter.scatter_plot(
        axs[2], x=t, y=photons, xaxislabel='time / s',
        yaxislabel='photons', facecolor='black'
    )
    for cp_idx in CPs[:-1]:
        if cp_idx < len(t):
            axs[2].axvline(x=t[cp_idx], color='black', ls='--', alpha=0.7, lw=1.5)
    # segment-mean photons
    for i in range(len(boundaries) - 1):
        seg_start = boundaries[i]
        seg_end = min(boundaries[i + 1], len(t))
        if seg_end <= seg_start:
            continue
        t_start = t[seg_start]
        t_end = t[seg_end - 1]
        axs[2].hlines(y=np.nanmean(photons[seg_start:seg_end]), xmin=t_start, xmax=t_end, color='grey', ls='-', lw=2, alpha=0.8)

    clear_output(wait=True)
    plt.show()
    plt.pause(5)

In [ ]:
# summary statistics
print("=" * 60)
print("Post-hoc analysis summary")
print("=" * 60)
print(f"Total datapoints loaded:              {len(df)}")
print(f"Removed (initial guess ~0.33):         {len(df) - len(df_filtered)}")
print(f"Total puncta:                          {df['puncta_id'].nunique()}")
print(f"Puncta with change points (A_R+A_G):   {len(cp_puncta_ids)}")
print(f"Datapoints in final dataset:           {len(df_cp)}")
print()
# breakdown of CP counts
n_cps = [len(cp_results[pid]) - 1 for pid in cp_puncta_ids]
print(f"Change points per punctum (joint): mean={np.mean(n_cps):.1f}, max={np.max(n_cps)}")

## Generate GIFs for puncta with change points

In [ ]:
from src import sCMOSFunctions
from colour_demosaicing import demosaicing_CFA_Bayer_bilinear
import tifffile

sCMOS = sCMOSFunctions.sCMOS_Functions()

# find TIFF associated with the H5 file
tiff_file = loc_files[file_idx].replace('.h5', '.tif')
if not os.path.exists(tiff_file):
    for ext in ['.tiff', '.TIF', '.TIFF']:
        alt = loc_files[file_idx].rsplit('.', 1)[0] + ext
        if os.path.exists(alt):
            tiff_file = alt
            break
print(f"TIFF file: {os.path.basename(tiff_file)}")

# load ROI metadata
metadata_files = H_F.file_search(dye_data_folder, "metadata", "")
x_coord, y_coord, roi_width, roi_height = IO.metadata_reader_imageJ(metadata_files[0])
print(f"ROI: x={x_coord}, y={y_coord}, w={roi_width}, h={roi_height}")

# memory-map the full TIFF (no copy into RAM)
with tifffile.TiffFile(tiff_file, is_ome=False, is_mmstack=False, is_imagej=False) as tif:
    raw_mmap = tif.asarray(out="memmap")
n_total_frames = raw_mmap.shape[0]
print(f"TIFF shape: {raw_mmap.shape}, frames: {n_total_frames}")
print(f"Using memory-mapped access — demosaicing will be done per-punctum.")

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from colour_demosaicing import demosaicing_CFA_Bayer_bilinear

# ─── Parameters ───
ROI_size = 12            # camera pixels (must be even)
half_roi = ROI_size // 2
upscale = 10             # super-resolution upscale factor
fps = 10                 # GIF frame rate
pixel_size = 69          # nm per camera pixel
exposure_time = 0.1      # seconds per frame
hr_pixel_size = pixel_size / upscale  # nm per reconstruction pixel
output_dir = os.path.join(dye_data_folder, "gifs")
os.makedirs(output_dir, exist_ok=True)


def render_fit_frame(shape_hr, xc_px, yc_px, sx_px, sy_px,
                     photons, A_R, A_G, A_B, upscale):
    """Render a super-resolved RGB frame from single-molecule fit parameters.

    Produces a coloured 2D Gaussian at the fitted sub-pixel position,
    with width from the fitted PSF and colour from spectral fractions.

    Args:
        shape_hr: (H, W) of high-resolution output in pixels.
        xc_px, yc_px: Centre position in camera pixel units (ROI-local).
        sx_px, sy_px: Gaussian sigma in camera pixels.
        photons: Total detected photoelectrons.
        A_R, A_G, A_B: Spectral fractions (sum ≈ 1).
        upscale: Integer upscale factor.

    Returns:
        (H, W, 3) float64 array — RGB intensities (unnormalised).
    """
    ys = np.arange(shape_hr[0]) / upscale
    xs = np.arange(shape_hr[1]) / upscale
    X, Y = np.meshgrid(xs, ys)

    sx = max(sx_px, 0.3)
    sy = max(sy_px, 0.3)

    gaussian = np.exp(-((X - xc_px)**2 / (2 * sx**2) +
                        (Y - yc_px)**2 / (2 * sy**2)))
    gaussian *= photons

    rgb = np.zeros((*shape_hr, 3))
    rgb[..., 0] = gaussian * A_R
    rgb[..., 1] = gaussian * A_G
    rgb[..., 2] = gaussian * A_B
    return rgb


# ─── Font (loaded once) ───
try:
    _font = ImageFont.truetype(
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 10)
except OSError:
    _font = ImageFont.load_default()

# ─── Generate GIFs ───
for pid in tqdm(sorted(cp_puncta_ids), desc="Creating GIFs"):
    subset = df_cp[df_cp['puncta_id'] == pid].sort_values('frame')
    if len(subset) < 5:
        continue

    # Stable centre from median position
    xc_med = subset['xc'].median()
    yc_med = subset['yc'].median()

    # ROI bounds — force even start for correct Bayer demosaicing
    y_start = max(0, int(np.round(yc_med)) - half_roi)
    x_start = max(0, int(np.round(xc_med)) - half_roi)
    if y_start % 2 != 0:
        y_start = max(0, y_start - 1)
    if x_start % 2 != 0:
        x_start = max(0, x_start - 1)
    y_end = min(raw_mmap.shape[1], y_start + ROI_size)
    x_end = min(raw_mmap.shape[2], x_start + ROI_size)

    actual_h = y_end - y_start
    actual_w = x_end - x_start
    hr_h = actual_h * upscale
    hr_w = actual_w * upscale

    # Pre-render all analysis frames (only frames with fit data)
    raw_list = []
    recon_list = []
    time_list = []

    for _, row in subset.iterrows():
        frame_idx = int(row['frame'])
        time_list.append(frame_idx * exposure_time)

        # Top panel: demosaic raw Bayer ROI → RGB
        raw_roi = raw_mmap[frame_idx, y_start:y_end,
                           x_start:x_end].astype(np.float64)
        rgb_raw = demosaicing_CFA_Bayer_bilinear(raw_roi)
        rgb_raw = np.clip(rgb_raw, 0, None)
        raw_list.append(rgb_raw)

        # Bottom panel: coloured Gaussian from fit parameters
        recon = render_fit_frame(
            (hr_h, hr_w),
            row['xc'] - x_start, row['yc'] - y_start,
            row['s_x'], row['s_y'],
            row['photons'], row['A_R'], row['A_G'], row['A_B'],
            upscale,
        )
        recon_list.append(recon)

    # Global intensity scaling (percentile-based)
    raw_all = np.array(raw_list)
    vmin_raw = np.percentile(raw_all, 1)
    vmax_raw = np.percentile(raw_all, 99.5)

    recon_all = np.array(recon_list)
    recon_pos = recon_all[recon_all > 0]
    vmax_recon = (np.percentile(recon_pos, 99.5)
                  if len(recon_pos) > 0 else 1.0)

    # Assemble GIF frames
    sep_h = 2                            # black separator between panels
    total_h = hr_h + sep_h + hr_h
    pil_frames = []

    for i in range(len(raw_list)):
        # Raw: normalise → uint8 → nearest-neighbour upscale
        raw_norm = np.clip(
            (raw_list[i] - vmin_raw) / (vmax_raw - vmin_raw + 1e-10), 0, 1)
        raw_u8 = (raw_norm * 255).astype(np.uint8)
        raw_up = np.repeat(
            np.repeat(raw_u8, upscale, axis=0), upscale, axis=1)

        # Reconstruction: normalise → uint8
        recon_norm = np.clip(
            recon_list[i] / (vmax_recon + 1e-10), 0, 1)
        recon_u8 = (recon_norm * 255).astype(np.uint8)

        # Stack: raw (top) | separator | fit (bottom)
        frame_arr = np.zeros((total_h, hr_w, 3), dtype=np.uint8)
        frame_arr[:hr_h] = raw_up
        frame_arr[hr_h + sep_h:] = recon_u8

        # Annotate via PIL
        img = Image.fromarray(frame_arr)
        draw = ImageDraw.Draw(img)

        draw.text((2, 2), "Raw", fill='white', font=_font)
        draw.text((2, hr_h + sep_h + 2), "Fit", fill='white', font=_font)

        # Time stamp (top-right)
        t_str = f"{time_list[i]:.1f} s"
        t_bbox = draw.textbbox((0, 0), t_str, font=_font)
        draw.text((hr_w - (t_bbox[2] - t_bbox[0]) - 3, 2),
                  t_str, fill='white', font=_font)

        # Scalebar on reconstruction panel
        bar_nm = 300
        bar_px = int(bar_nm / hr_pixel_size)
        margin = 4
        bar_y = total_h - margin - 2
        bar_x_end = hr_w - margin
        bar_x_start = bar_x_end - bar_px
        draw.rectangle([bar_x_start, bar_y, bar_x_end, bar_y + 2],
                       fill='white')
        lbl = f"{bar_nm} nm"
        lbl_bbox = draw.textbbox((0, 0), lbl, font=_font)
        lbl_w = lbl_bbox[2] - lbl_bbox[0]
        draw.text((bar_x_start + (bar_px - lbl_w) // 2, bar_y - 13),
                  lbl, fill='white', font=_font)

        pil_frames.append(img)

    # Write GIF via PIL (much faster than matplotlib FuncAnimation)
    gif_path = os.path.join(output_dir, f"puncta_{pid}.gif")
    pil_frames[0].save(
        gif_path, save_all=True, append_images=pil_frames[1:],
        duration=1000 // fps, loop=0, optimize=False,
    )

print(f"\nSaved {len(cp_puncta_ids)} GIFs to {output_dir}")